# 🛡️ 第 13 课 · Guardrails 护栏速成

> **护栏速成**：在 Agent 前后加输入过滤、PII、HITL、输出安全校验，防止不安全行为。

本课你会学到：
1. Guardrails 是什么、为什么重要
2. 确定性规则 vs 模型判定两种思路
3. 内置：PII Middleware / HITL Middleware
4. 自定义：Before-Agent / After-Agent 护栏
5. 多层组合 + 医疗聊天机器人案例

模型：**DeepSeek V4 Pro**（`deepseek-v4-pro`）。`.env` 需 `DEEPSEEK_API_KEY`（可选 `DEEPSEEK_BASE_URL`）。

> 📌 Docs：https://docs.langchain.com/oss/python/langchain/guardrails


本笔记涵盖在 Agent 系统中实现 **Guardrails（护栏）** 所需的全部要点。

### 📚 涵盖主题
1. 什么是 Guardrails？为什么重要？
2. 两种思路：确定性规则 vs 模型判定
3. 内置：PII（个人身份信息）检测中间件
4. 内置：Human-in-the-Loop（人在回路）中间件
5. 自定义：Before-Agent 护栏（输入过滤）
6. 自定义：After-Agent 护栏（输出安全）
7. 分层 / 组合护栏
8. 实战案例：医疗聊天机器人

---
> 📌 **文档参考：** https://docs.langchain.com/oss/python/langchain/guardrails


## 逻辑总览（护栏层）

本课使用 **DeepSeek V4 Pro**（`deepseek-v4-pro`）。`.env` 需 `DEEPSEEK_API_KEY`（可选 `DEEPSEEK_BASE_URL`）。

```mermaid
%%{init: {
  "theme": "base",
  "themeVariables": {
    "fontSize": "13px",
    "fontFamily": "ui-sans-serif, system-ui",
    "primaryTextColor": "#0f172a",
    "lineColor": "#94a3b8"
  }
}}%%
flowchart TB
    U([用户输入]) --> BI[Before-Agent<br/>输入过滤]
    BI -->|拦截| BLK([拒绝 / 提示])
    BI -->|放行| AG[Agent + Tools<br/>deepseek-v4-pro]
    AG --> PII[PII Middleware]
    AG --> HITL[HITL Middleware]
    PII --> AO[After-Agent<br/>输出校验]
    HITL --> AO
    AO -->|不安全| BLK
    AO -->|安全| OUT([返回用户])


    classDef input fill:#BFDBFE,stroke:#3B82F6,color:#1E3A8A,stroke-width:2px
    classDef llm fill:#FED7AA,stroke:#F97316,color:#9A3412,stroke-width:2px
    classDef branch fill:#E9D5FF,stroke:#A855F7,color:#6B21A8,stroke-width:2px
    classDef tool fill:#BBF7D0,stroke:#22C55E,color:#14532D,stroke-width:2px
    classDef output fill:#FECACA,stroke:#F87171,color:#7F1D1D,stroke-width:2px
    class U input
    class BI,AO branch
    class AG llm
    class PII,HITL tool
    class BLK,OUT output
```


**要点：** 护栏是中间件，夹在 Agent 前后；确定性规则快，模型审核抓语义。



---
## 🧠 第 1 节：什么是 Guardrails？

Guardrails（护栏）是控制 AI Agent **输入与输出** 的安全机制。
它们包裹在 Agent 流水线外围，确保 Agent：

* 只处理安全、合适的输入
* 只执行已批准的操作
* 只返回经过校验、符合合规要求的输出

通过在 Agent 执行的关键节点做内容校验与过滤，护栏帮助你构建 **安全、合规的 AI 应用**。

它们以 **中间件（middleware）** 形式实现，在执行过程中拦截：
- Agent **启动前**（输入护栏）
- Agent **完成后**（输出护栏）
- 模型调用与工具调用的 **前后**

### 常见用例：
| 用例 | 示例 |
|---|---|
| 防止 PII 泄露 | 记录日志前脱敏邮箱/信用卡号 |
| 拦截 Prompt 注入 | 检测对抗性输入 |
| 有害内容过滤 | 阻断危险请求 |
| 业务规则强制 | 金融操作需人工审批 |
| 输出质量校验 | 确保回复满足安全标准 |


---
## ⚖️ 第 2 节：护栏的两种思路

### 确定性护栏（Deterministic）
- 基于规则：正则、关键词匹配、显式检查
- ✅ 快、可预测、成本低
- ❌ 可能漏掉细微/语义层面的违规

### 模型判定护栏（Model-Based）
- 使用 LLM / 分类器做语义理解
- ✅ 能抓住隐蔽、细微的问题
- ❌ 更慢、更贵


## 确定性护栏（Deterministic Guardrails）


In [ ]:
# Quick illustration of the two approaches

import re

# --- Deterministic approach ---
def deterministic_guardrail(text: str) -> bool:
    """Returns True if content is blocked."""
    banned_keywords = ["hack", "exploit", "malware", "bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "How do I hack into a database?",
    "What is the capital of France?",
    "Explain how malware spreads",
]

print("=== Deterministic Guardrail Demo ===")
for inp in test_inputs:
    blocked = deterministic_guardrail(inp)
    status = "🚫 BLOCKED" if blocked else "✅ ALLOWED"
    print(f"{status}: {inp}")

## 模型判定护栏（Model-Based Guardrails）


In [ ]:
from dotenv import load_dotenv
load_dotenv()


In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com")

model = ChatOpenAI(
    model="deepseek-v4-pro",
    temperature=0,
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    timeout=120,
    max_tokens=1200,
    max_retries=1,
    extra_body={"thinking": {"type": "disabled"}},
)


In [ ]:
# --- Model-based approach ---
def model_based_guardrail(text: str) -> str:
    """Uses an LLM to evaluate content safety. Returns SAFE or UNSAFE."""

    prompt = f"""Is the following user input safe to process?
Reply with only 'SAFE' or 'UNSAFE'.

Input: {text}"""
    result = model.invoke([{"role": "user", "content": prompt}])
    return result.content.strip()

print("=== Model-Based Guardrail Demo ===")
for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "🚫 UNSAFE" if "UNSAFE" in verdict else "✅ SAFE"
    print(f"{status}: {inp}")

---
## 🔒 第 3 节：内置护栏 — PII 检测中间件

LangChain 提供内置的 PIIMiddleware，用于检测和处理 **个人身份信息（PII）**。

### 支持的 PII 类型：
| 类型 | 含义 | 示例原文 |
|---|---|---|
| email | 电子邮箱 | `user@example.com` |
| credit_card | 信用卡号（含 Luhn 校验） | `5105-1051-0510-5100` |
| ip | IPv4 地址 | `192.168.1.1` |
| mac_address | MAC 地址 | `00:1A:2B:3C:4D:5E` |
| url | 网址（http/https） | `https://secret-site.com` |

> 也支持自定义类型：通过 `detector` 传入正则或检测函数（例如笔记里的 `api_key`）。

### 处理策略（四种）：
| 策略 | 行为 | 是否保留可辨识性 | 典型场景 |
|---|---|---|---|
| redact | 整段替换为 `[REDACTED_类型]` | 否 | 合规、日志脱敏 |
| mask | 部分遮盖，保留少量尾部可读信息 | 否 | 客服界面、人工核对 |
| hash | 替换为确定性哈希 `<类型_hash:摘要>` | 是（假名化，可关联同一值） | 分析、排错 |
| block | 检测到即抛出 `PIIDetectionError` | N/A | 严禁出现该类信息 |

### 各类型具体会怎么处理：

| 类型 | 原文 | redact | mask | hash | block |
|---|---|---|---|---|---|
| email | `user@example.com` | `[REDACTED_EMAIL]` | `user@****.com`（保留用户名，域名打码） | `<email_hash:a1b2c3d4>` | 抛异常 |
| credit_card | `5105-1051-0510-5100` | `[REDACTED_CREDIT_CARD]` | `****-****-****-5100`（仅留后 4 位） | `<credit_card_hash:…>` | 抛异常 |
| ip | `192.168.1.1` | `[REDACTED_IP]` | `*.*.*.1`（仅留最后一段） | `<ip_hash:…>` | 抛异常 |
| mac_address | `00:1A:2B:3C:4D:5E` | `[REDACTED_MAC_ADDRESS]` | `**:**:**:**:**:5E`（仅留末 2 位） | `<mac_address_hash:…>` | 抛异常 |
| url | `https://secret-site.com` | `[REDACTED_URL]` | `[MASKED_URL]`（整段遮盖） | `<url_hash:…>` | 抛异常 |

**本课示例配置：**
- `email` → `strategy="redact"`：模型侧看不到真实邮箱
- `credit_card` → `strategy="mask"`：只留卡号后 4 位便于沟通
- `api_key`（自定义正则）→ `strategy="block"`：一出现密钥直接拦截


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool

In [ ]:
# Define a simple dummy tool
@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    print(f"customer_lookup enter")
    return f"Customer record found for query: {query}"


# Create agent with PII Middleware
agent = create_agent(
    model=model,
    tools=[customer_lookup],
    middleware=[
        # Redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # Block API keys - raise error if detected
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

print("Agent with PII middleware created successfully!")


In [ ]:
# Test PII Redaction
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"
    }]
})



In [ ]:
print("=== Agent Response ===")
print(result["messages"][-1].content)

In [ ]:
result

In [ ]:
# Test API Key Blocking
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })

except Exception as e:
    print(f"🚫 Blocked as expected: {e}")

In [ ]:
result

---
## 👤 第 4 节：内置护栏 — Human-in-the-Loop 中间件

在敏感操作执行前暂停 Agent，等待人工审批。

**适用于：**
- 金融交易
- 向外部发送邮件
- 删除生产数据
- 任何有重大业务影响的操作

**关键要求：** 需要 checkpointer，以便在中断期间持久化状态。


In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool



In [ ]:
@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"



# Create agent with HITL middleware
hitl_agent = create_agent(
    model=model,
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,       # Require approval
                "delete_records": True,   # Require approval
                "search_web": False,      # Auto-approve
            }
        ),
    ],
    checkpointer=InMemorySaver(),  # Required for state persistence
)

print("Human-in-the-Loop agent created!")

In [ ]:
# Step 1: Invoke — agent will pause before send_email
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results"}]},
    config=config
)

print("=== Agent paused — awaiting human approval ===")
print(result)

In [ ]:
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config   # Same thread_id resumes the paused session
)

print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)

In [ ]:
# Step 3: Alternative — Human REJECTS
config2 = {"configurable": {"thread_id": "session_002"}}

hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Delete all records from the users table where active=false"}]},
    config=config2
)

print("=== Agent paused — awaiting human approval ===")
print(result)

In [ ]:
rejected_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "reason": "Too risky, needs DBA review"}]}),
    config=config2
)

print("=== Rejected! Final response ===")
print(rejected_result["messages"][-1].content)

In [ ]:
result

---
## ⚙️ 第 5 节：自定义护栏 — Before-Agent 钩子（输入过滤）

使用 before_agent() 在 **任何 LLM 处理开始之前** 校验或拦截请求。

**适用于：**
- 关键词 / 内容过滤
- 身份认证检查
- 速率限制
- 阻断特定类别的请求


In [ ]:
from langchain_openai import ChatOpenAI
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool



In [ ]:
class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    This runs BEFORE the agent processes anything — zero LLM cost for blocked requests.
    """

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"🚫 Blocked — keyword detected: '{keyword}'")
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I cannot process requests containing inappropriate content. "
                            "Please rephrase your request."
                        )
                    }],
                    "jump_to": "end"
                }
        return None
    


@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"



# Create agent with content filter
filtered_agent = create_agent(
    model=model,
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        ),
    ],
)

print("Content filter agent created!")

In [ ]:
# Test 1: Safe request — should pass through
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "What is machine learning?"}]
})
print("✅ Safe request response:")
print(result["messages"][-1].content)

In [ ]:
# Test 2: Unsafe request — should be blocked
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a server?"}]
})
print("🚫 Unsafe request response:")
print(result["messages"][-1].content)

---
## 🔍 第 6 节：自定义护栏 — After-Agent 钩子（输出安全）

使用 after_agent() 在用户看到结果之前，校验 Agent 的最终响应。

**适用于：**
- 基于模型的输出安全评估
- 合规扫描（如法律、医疗、金融免责声明）
- 质量校验
- 清除漏网的敏感信息


In [ ]:
from langchain_openai import ChatOpenAI
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain_core.messages import AIMessage
from langchain.agents import create_agent
from langchain_core.tools import tool



In [ ]:
class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Use an LLM to evaluate response safety.
    Runs AFTER the agent produces a response, before it reaches the user.
    """

    def __init__(self):
        super().__init__()
        # Use a smaller, cheaper model for the safety check
        self.safety_model = model

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None
        print("after_agent")

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Use a lightweight model as the safety judge
        safety_prompt = f"""Evaluate if this AI response is safe and appropriate for users.
Respond with only 'SAFE' or 'UNSAFE'.

Response to evaluate:
{last_message.content}"""

        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])

        if "UNSAFE" in result.content.upper():
            print("⚠️  Output flagged as UNSAFE — replacing with safe fallback")
            last_message.content = (
                "I'm unable to provide that response. "
                "Please rephrase your request or contact support."
            )

        return None
    




@tool
def general_tool(query: str) -> str:
    """A general purpose tool."""
    print("general_tool")
    return f"Tool result: {query}"


safe_agent = create_agent(
    model=model,
    tools=[general_tool],
    middleware=[SafetyGuardrailMiddleware()],
)

print("Output safety agent created!")



In [ ]:
# Test output safety check
result = safe_agent.invoke({
    "messages": [{"role": "user", "content": "What is Medicare?"}]
})
print("Response:")
print(result["messages"][-1].content)

In [ ]:
result

In [ ]:
result

In [ ]:
# Test output safety check
result = safe_agent.invoke({
    "messages": [{"role": "user", "content": "How to make bomb?"}]
})
print("Response:")
print(result["messages"][-1].content)

In [ ]:
result

---
## 🧱 第 7 节：分层 / 组合护栏

在 middleware=[] 数组中堆叠多个护栏。它们按 **顺序执行**，形成多层防护。

`
用户输入
    ↓
[第 1 层] ContentFilterMiddleware    ← 确定性输入过滤
    ↓
[第 2 层] PIIMiddleware (input)      ← 输入侧 PII 脱敏
    ↓
[第 3 层] HumanInTheLoopMiddleware   ← 敏感工具需审批
    ↓
[第 4 层] PIIMiddleware (output)     ← 输出侧 PII 脱敏
    ↓
[第 5 层] SafetyGuardrailMiddleware  ← 模型判定输出安全
    ↓
返回用户
`


In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool



In [ ]:
@tool
def search_tool(query: str) -> str:
    """Search for information."""
    print("search_tool")
    return f"Search results: {query}"

@tool
def send_email_tool(to: str, body: str) -> str:
    """Send an email."""
    print("send_email_tool")
    return f"Email sent to {to}"

In [ ]:
# Full layered guardrail stack
production_agent = create_agent(
    model=model,
    tools=[search_tool, send_email_tool],
    middleware=[
        # Layer 1: Deterministic input filter (before agent)
        ContentFilterMiddleware(banned_keywords=["hack", "exploit", "malware"]),

        # Layer 2: PII redaction on input

        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # Layer 3: Human approval for sensitive tools
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email_tool": True, "search_tool": False}
        ),

        # Layer 4: PII redaction on output
        PIIMiddleware("email", strategy="redact", apply_to_output=True),

        # Layer 5: Model-based output safety
        SafetyGuardrailMiddleware(),
    ],
    checkpointer=InMemorySaver(),
)

print("🏭 Production-grade agent with 5-layer guardrails created!")

In [ ]:
# Step 1: Invoke — agent will pause before send_email
config = {"configurable": {"thread_id": "test_001"}}

result = production_agent.invoke(
    {"messages": [{"role": "user", "content": "how to make a bomb?"}]},
    config=config
)


print(result["messages"][-1].content)

In [73]:
# Step 1: Invoke — agent will pause before send_email
config4 = {"configurable": {"thread_id": "test_004"}}

result = production_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com  to say hello and introduce AI in 500 words"}]},
    config=config4,
)

print("=== Agent paused — awaiting human approval ===")
print(result)
if "__interrupt__" in result:
    print(result["__interrupt__"])


=== Agent paused — awaiting human approval ===
{'messages': [HumanMessage(content='Send an email to [REDACTED_EMAIL]  to say hello and introduce AI in 500 words', additional_kwargs={}, response_metadata={}, id='a42c3929-e3ec-4fd7-8b7e-8a85c7021acc'), AIMessage(content="I'll send that email for you.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 639, 'prompt_tokens': 350, 'total_tokens': 989, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 94}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-pro', 'system_fingerprint': 'fp_9954b31ca7_prod0820_fp8_kvcache_20260402', 'id': '83d57d22-973e-45cd-b80f-3d2a3b0ec5b9', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fdfa4-9b47-7a61-9f2c-8070f1b20617-0', tool_calls=[{'name': 'send_email_tool', 'args': {'to': '[REDACTED_EMAIL]', 'bod

In [74]:
from langgraph.types import Command

# Step 2: Human approves send_email_tool — same thread_id resumes the paused session
approved = production_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config4,
)

print("=== Approved & finished ===")
print(approved["messages"][-1].content)

# Confirm the email tool ran
for m in approved["messages"]:
    if getattr(m, "name", None) == "send_email_tool" or (
        m.__class__.__name__ == "ToolMessage" and "Email sent" in str(getattr(m, "content", ""))
    ):
        print("ToolMessage:", m.content)


send_email_tool
after_agent
=== Approved & finished ===
The email has been sent successfully to [REDACTED_EMAIL]. It includes a friendly greeting and a comprehensive ~500-word introduction to artificial intelligence, covering:

- **What AI is** — the simulation of human intelligence by machines
- **Types of AI** — Narrow AI, General AI, and Superintelligent AI
- **How it works** — machine learning, deep learning, and neural networks
- **Real-world applications** — healthcare, finance, transportation, education, and creative fields
- **Ethical considerations** — bias, privacy, job displacement, and accountability

Let me know if you'd like any adjustments!
ToolMessage: Email sent to [REDACTED_EMAIL]


---
## 🏥 第 8 节：实战案例 — 医疗聊天机器人

一个医疗聊天机器人，能够：
1. **拦截** 跑题或有害请求
2. **脱敏** 患者 PII（邮箱、信用卡号等）
3. **预约前需人工审批**
4. **校验** 输出在医学上是否合适


In [ ]:
from langchain_openai import ChatOpenAI
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import AIMessage



In [75]:
# --- Healthcare-specific content filter ---
class HealthcareSafetyFilter(AgentMiddleware):
    """Block non-medical or harmful requests in a healthcare context."""

    BLOCKED_TOPICS = ["drug synthesis", "self-harm", "suicide method", "weapon", "hack"]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_msg = state["messages"][0]
        if first_msg.type != "human":
            return None

        content = first_msg.content.lower()
        for topic in self.BLOCKED_TOPICS:
            if topic in content:
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I'm a healthcare assistant and can only help with "
                            "medical questions, appointments, and health information. "
                            "If you're in crisis, please call 112 or your local emergency number."
                        )
                    }],
                    "jump_to": "end"
                }
        return None
    



# --- Medical output validator ---
class MedicalOutputValidator(AgentMiddleware):
    """Ensure all responses include appropriate medical disclaimers."""

    DISCLAIMER = "\n\n⚕️ *This is general health information, not medical advice. Please consult a qualified healthcare professional.*"

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Add disclaimer if not already present
        if "medical advice" not in last_message.content.lower():
            last_message.content += self.DISCLAIMER

        return None
    


# --- Healthcare tools ---
@tool
def search_symptoms(symptoms: str) -> str:
    """Search for information about medical symptoms."""
    return f"Symptom information for: {symptoms}. Please consult a doctor for diagnosis."

@tool
def book_appointment(patient_name: str, date: str, doctor: str) -> str:
    """Book a medical appointment."""
    return f"Appointment booked for {patient_name} with Dr. {doctor} on {date}"

@tool
def get_medication_info(medication: str) -> str:
    """Get information about a medication."""
    return f"General info about {medication}. Always follow your doctor's prescription."




# --- Build the healthcare chatbot ---
healthcare_bot = create_agent(
    model=model,
    tools=[search_symptoms, book_appointment, get_medication_info],
    middleware=[
        # Guardrail 1: Block harmful/off-topic requests
        HealthcareSafetyFilter(),

        # Guardrail 2: Redact patient PII from inputs
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # Guardrail 3: Require approval before booking appointments
        HumanInTheLoopMiddleware(
            interrupt_on={
                "book_appointment": True,
                "search_symptoms": False,
                "get_medication_info": False,
            }
        ),

        # Guardrail 4: Add medical disclaimer to all outputs
        MedicalOutputValidator(),
    ],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "You are a helpful healthcare assistant. "
        "You can search for symptoms, medication information, and help book appointments. "
        "Always be empathetic and remind users to consult a doctor for diagnosis."
    )
)

print("🏥 Healthcare chatbot with full guardrail stack created!")

🏥 Healthcare chatbot with full guardrail stack created!


In [77]:
# Test 1: Safe medical query
config_t1 = {"configurable": {"thread_id": "healthcare_session_t1"}}

result = healthcare_bot.invoke(
    {"messages": [{"role": "user", "content": "What are symptoms of Type 2 Diabetes?"}]},
    config=config_t1
)

print(result["messages"][-1].content)

Here are the common symptoms associated with **Type 2 Diabetes**:

### Common Symptoms:
- **Increased thirst** (polydipsia) and dry mouth
- **Frequent urination** (polyuria), especially at night
- **Increased hunger** (polyphagia)
- **Unexplained weight loss**, even when eating normally
- **Fatigue** and feeling unusually tired
- **Blurred vision**
- **Slow-healing sores or cuts** and frequent infections
- **Tingling, numbness, or pain** in the hands or feet (neuropathy)
- **Darkened skin patches**, often in the armpits or neck (acanthosis nigricans)

### Important Notes:
- Symptoms often develop **gradually** over time, and some people may not notice them for years.
- Many individuals with Type 2 Diabetes may have no symptoms at all initially, which is why regular check-ups are important, especially if you have risk factors like being overweight, having a family history, or leading a sedentary lifestyle.

⚠️ **Please remember:** I'm here to provide general information, but I'm not a d

In [79]:
# 方式 1：转成 list
history = list(healthcare_bot.get_state_history(config_t1))
for i, snap in enumerate(history):
    print(f"--- history[{i}] step={snap.metadata.get('step')} next={list(snap.next)} ---")
    print(snap.values)

--- history[0] step=15 next=[] ---
{'messages': [HumanMessage(content='What are symptoms of Type 2 Diabetes?', additional_kwargs={}, response_metadata={}, id='9369d66b-d861-40d6-83c8-fb9962d9ea26'), AIMessage(content="I'd be happy to help you learn about the symptoms of Type 2 Diabetes. Let me look that up for you.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 441, 'total_tokens': 514, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 384}, 'prompt_cache_hit_tokens': 384, 'prompt_cache_miss_tokens': 57}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-pro', 'system_fingerprint': 'fp_9954b31ca7_prod0820_fp8_kvcache_20260402', 'id': 'd22317fc-60dc-4571-b69d-d7d1767c886c', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fdfa9-ddce-7cf2-9537-33becefcc67d-0', tool_calls=[{'name': 'search_symptoms', 'args': {'symptoms': 'T

In [80]:
# Test 2: Query with PII (email gets redacted)
result = healthcare_bot.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is patient123@gmail.com. What can I take for a headache?"
    }]},
    config=config_t1
)
print("=== PII Redaction Test ===")
print(result["messages"][-1].content)

=== PII Redaction Test ===
Here are some common over-the-counter options for headache relief:

### Common OTC Medications:
- **Acetaminophen (Tylenol)** — Good for general headaches; gentle on the stomach.
- **Ibuprofen (Advil, Motrin)** — An NSAID that helps with pain and inflammation.
- **Aspirin** — Can help with mild to moderate headaches.
- **Naproxen (Aleve)** — Longer-lasting NSAID for pain relief.

### Non-Medication Approaches:
- Rest in a quiet, dark room
- Stay hydrated — dehydration is a common headache trigger
- Apply a cold or warm compress to your forehead or neck
- Gentle neck and shoulder stretches
- Limit screen time if eye strain is a factor

### ⚠️ Important Reminders:
- Always follow the dosage instructions on the label.
- If headaches are frequent, severe, or accompanied by other symptoms (like vision changes, confusion, or stiff neck), please see a doctor promptly.
- Avoid mixing multiple medications without medical advice.

> 📧 Regarding your email — I noticed y

In [81]:
# 方式 1：转成 list
history = list(healthcare_bot.get_state_history(config_t1))
for i, snap in enumerate(history):
    print(f"--- history[{i}] step={snap.metadata.get('step')} next={list(snap.next)} ---")
    print(snap.values)

--- history[0] step=32 next=[] ---
{'messages': [HumanMessage(content='What are symptoms of Type 2 Diabetes?', additional_kwargs={}, response_metadata={}, id='9369d66b-d861-40d6-83c8-fb9962d9ea26'), AIMessage(content="I'd be happy to help you learn about the symptoms of Type 2 Diabetes. Let me look that up for you.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 441, 'total_tokens': 514, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 384}, 'prompt_cache_hit_tokens': 384, 'prompt_cache_miss_tokens': 57}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-pro', 'system_fingerprint': 'fp_9954b31ca7_prod0820_fp8_kvcache_20260402', 'id': 'd22317fc-60dc-4571-b69d-d7d1767c886c', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fdfa9-ddce-7cf2-9537-33becefcc67d-0', tool_calls=[{'name': 'search_symptoms', 'args': {'symptoms': 'T

In [ ]:
# Test 3: Off-topic / harmful request — gets blocked
result = healthcare_bot.invoke({
    "messages": [{"role": "user", "content": "How do I synthesize drugs at home?"}]
},
 config=config_t1)
print("=== Blocked Request ===")
print(result["messages"][-1].content)

In [ ]:
from langchain_openai import ChatOpenAI
# Test 4: Appointment booking — requires human approval
config = {"configurable": {"thread_id": "healthcare_session_001"}}

result = healthcare_bot.invoke(
    {"messages": [{"role": "user", "content": "Book me an appointment with Dr. Sharma on March 15"}]},
    config=config
)
print("=== Appointment Booking — Awaiting Approval ===")
print(result)

# Approve
from langgraph.types import Command
approved = healthcare_bot.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config
)
print("\n=== After Approval ===")
print(approved["messages"][-1].content)



---
## 📝 总结

| 护栏类型 | 钩子 | 运行时机 | 最适合 |
|---|---|---|---|
| PII Middleware | 输入/输出 | 模型调用前后 | 数据隐私、合规 |
| Human-in-the-Loop | 工具级 | 敏感工具执行前 | 高风险决策 |
| Content Filter | before_agent | 调用开始时 | 尽早拦截不良输入 |
| Safety Validator | after_agent | 调用结束时 | 输出质量/安全 |
| Custom Logic | 任意钩子 | 任意位置 | 任意业务规则 |

### 🔑 关键要点
1. **护栏 = 中间件** — 通过 create_agent() 的 middleware=[] 参数接入
2. **分层护栏** — 纵深防御是最佳实践
3. **先确定性、后模型判定** — 尽早用廉价规则检查，避免不必要的昂贵 LLM 调用
4. **HITL 需要 checkpointer** — 开发用 InMemorySaver，生产用持久化存储
5. **自定义中间件** — 通过 before_agent() / after_agent() 获得完整控制权

---
### 📚 更多资源
- [LangChain Guardrails 文档](https://docs.langchain.com/oss/python/langchain/guardrails)
- [Middleware 文档](https://docs.langchain.com/oss/python/langchain/middleware/overview)
- [Human-in-the-Loop 文档](https://docs.langchain.com/oss/python/langchain/human-in-the-loop)
- [LangSmith 可观测性](https://docs.langchain.com/oss/python/langchain/observability)

---
